# Export Gebäude Stadt Luzern (Baujahr 1970–2000)

Dieses Notebook dokumentiert die Einzelschritte zum Download, Filtern und Exportieren der Datensätze aus:
https://daten.geo.lu.ch/browser/#/collections/KGWRPUBL_COL_V3/
Die Arbeitsstruktur ist jetzt getrennt in `notebook/` und `output/`.

## 1) Datenquelle und lokale Pfade

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import urllib.request
import zipfile

STAC_ITEM_URL = 'https://daten.geo.lu.ch/api/stac/v1.0/collections/KGWRPUBL_COL_V3/items?limit=1'
GPKG_ZIP_URL = 'https://download.geo.lu.ch/api/stac/v1.0/downloads/KGWRPUBL_COL_V3/KGWRPUBL_COL/KGWRPUBL_COL_V3_gpkg.zip'

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebook' else cwd
base = project_root / 'output' / 'kgwr'
zip_path = base / 'KGWRPUBL_COL_V3_gpkg.zip'
extract_dir = base
gpkg_path = base / 'daten' / 'KGWRGPUB_DS_V3_20260812.gpkg'
csv_path = base / 'kgwr_gebaeude_luzern_baujahr_1970_2000.csv'

base.mkdir(parents=True, exist_ok=True)
print('Basisordner:', base.resolve())

Basisordner: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr


## 2) GeoPackage herunterladen und entpacken

In [3]:
urllib.request.urlretrieve(GPKG_ZIP_URL, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print('ZIP:', zip_path.resolve())
print('GPKG vorhanden:', gpkg_path.exists())

ZIP: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr/KGWRPUBL_COL_V3_gpkg.zip
GPKG vorhanden: True


## 3) Daten mit pandas auf Stadt Luzern und Baujahr 1970–2000 filtern

In [4]:
gdf = gpd.read_file(gpkg_path, layer='KGWRGPUB_V3_PT')
filtered = gdf[(gdf['BFS_GEMEINDE'] == 'Luzern') & (gdf['GBAUJ'].between(1970, 2000))].copy()
filtered = filtered.sort_values('EGID')

print('Gefilterte Datensätze:', len(filtered))

/Users/padrian/miniconda3/envs/georg-moersch-bildanalyse/lib/python3.9/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 111206 in column GWAERDATH1, 2024-06-04T00:00:00.0Z, successfully parsed
  return ogr_read(


Gefilterte Datensätze: 2406


## 4) Ergebnis als CSV exportieren

In [10]:
filtered.drop(columns='geometry', errors='ignore').to_csv(csv_path, index=False, encoding='utf-8')
print('CSV geschrieben:', csv_path.resolve())

CSV geschrieben: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr/kgwr_gebaeude_luzern_baujahr_1970_2000.csv


## 5) Plausibilitätscheck

In [5]:
df = pd.read_csv(csv_path)

print('Zeilen:', len(df))
print('Baujahr min/max:', int(df['GBAUJ'].min()), int(df['GBAUJ'].max()))
print('Gemeinden:', sorted(df['BFS_GEMEINDE'].dropna().unique())[:5])
print('Erste 5 Zeilen:')
display(df.head(5))


Zeilen: 2406
Baujahr min/max: 1970 2000
Gemeinden: ['Luzern']
Erste 5 Zeilen:


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GWAERDATW1,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB
0,208349,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,22.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
1,208350,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,24.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
2,208351,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,26.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
3,208352,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,28.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
4,208353,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,30.0,NaN,NaN,...,2018-11-08 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00


## 6) Datenanalyse

Wir können nach Strassennamen suchen

In [6]:
alpenquai = filtered[filtered['STRNAMK1_HPT'].astype(str).str.contains('Alpenquai', case=False, na=False)].copy()

print('Treffer:', len(alpenquai))
display(alpenquai[['STRNAMK1_HPT', 'DEINR', 'GKLAS_TXT', 'GBEZ']])

Treffer: 26


,STRNAMK1_HPT,DEINR,GKLAS_TXT,GBEZ
27636,Alpenquai,12,Gross- und Einzelhandel,None
27637,Alpenquai,14,Bürogebäude,None
27710,Alpenquai,40,Gebäude mit 3+ Whgen,None
27713,Alpenquai,34,Gebäude mit 3+ Whgen,None
27714,Alpenquai,34,Gebäude mit 3+ Whgen,None
27726,Alpenquai,34,Gebäude mit 3+ Whgen,None
27727,Alpenquai,34,Gebäude mit 3+ Whgen,None
72269,Alpenquai,30,Bürogebäude,None
27732,Alpenquai,33,Sporthalle,Bootshaus
76514,Alpenquai,11,Industriegebäude,Bootsreparatur- Werkstätte SNG


In der Spalte GKLAS_TXT werden die Gebäude Nutzunge zugeschrieben. Im nächsten Schritt sammeln wir alle Werte dieser Spalte, um einen Überblick zu erhalten.

In [8]:
unique_gklas_txt = sorted(filtered['GKLAS_TXT'].dropna().unique())

display(unique_gklas_txt)

['Andere Beherbergung',
 'Andere landw. Geb.',
 'Behälter, Silo, Lager',
 'Bürogebäude',
 'Garagengebäude',
 'Gebäude mit 1 Wohnung',
 'Gebäude mit 2 Wohnungen',
 'Gebäude mit 3+ Whgen',
 'Gross- und Einzelhandel',
 'Hotelgebäude',
 'Industriegebäude',
 'Kirche / Kultgebäude',
 'Krankenhaus',
 'Kultur-/Freizeitstätte',
 'Landw. Betriebsgebäude',
 'Museum / Bibliothek',
 'Pflanzenbau',
 'Schul-/Hochschulgebäude',
 'Sonstiger Hochbau',
 'Sporthalle',
 'Verkehr / Kommunikation',
 'Wohngeb.f.Gemeinschaften']

Wieviele Gebäude werden den jeweiligen Nutzunge zugeordnet?

In [19]:
gklas_counts = filtered['GKLAS_TXT'].value_counts(dropna=True)
display(gklas_counts)

GKLAS_TXT
Gebäude mit 3+ Whgen        714
Garagengebäude              629
Gebäude mit 1 Wohnung       278
Sonstiger Hochbau           163
Industriegebäude            120
Bürogebäude                  91
Behälter, Silo, Lager        70
Kultur-/Freizeitstätte       68
Gebäude mit 2 Wohnungen      67
Landw. Betriebsgebäude       48
Schul-/Hochschulgebäude      37
Verkehr / Kommunikation      25
Wohngeb.f.Gemeinschaften     22
Gross- und Einzelhandel      20
Kirche / Kultgebäude         14
Krankenhaus                  13
Sporthalle                    9
Hotelgebäude                  8
Museum / Bibliothek           5
Andere Beherbergung           3
Andere landw. Geb.            1
Pflanzenbau                   1
Name: count, dtype: int64

Nun können wir nach Nutzungen filtern

In [9]:
kirche = filtered[filtered['GKLAS_TXT'] == 'Kirche / Kultgebäude'].copy()

print('Kirche-/Kultgebäude:', len(kirche))
display(kirche[['STRNAMK1_HPT', 'DEINR', 'GKLAS_TXT', 'GBEZ']])

Kirche-/Kultgebäude: 14


,STRNAMK1_HPT,DEINR,GKLAS_TXT,GBEZ
71511,Blattenmoosstrasse,8,Kirche / Kultgebäude,None
26044,Spitalstrasse,91,Kirche / Kultgebäude,None
26889,Winkelriedstrasse,5,Kirche / Kultgebäude,Pfarreiheim Barfüesser
27734,Langensandstrasse,1,Kirche / Kultgebäude,Pfarreiheim
75650,Zollhausstrasse,5,Kirche / Kultgebäude,None
80038,Eichenstrasse,23,Kirche / Kultgebäude,Friedhofhalle
85606,St.-Leodegar-Strasse,6,Kirche / Kultgebäude,Pfarrei-Saal St Leodegar
86097,Ibachstrasse,2,Kirche / Kultgebäude,Leichenhaus
86615,Landschaustrasse,6,Kirche / Kultgebäude,Seelsorgestation ABZUBRECHEN
86735,Luzernerstrasse,90,Kirche / Kultgebäude,Kirche


In [10]:
museum = filtered[filtered['GKLAS_TXT'] == 'Museum / Bibliothek'].copy()

print('Museum / Bibliothek:', len(museum))
display(museum[['STRNAMK1_HPT', 'DEINR', 'GKLAS_TXT', 'GBEZ']])

Museum / Bibliothek: 5


,STRNAMK1_HPT,DEINR,GKLAS_TXT,GBEZ
79313,Lidostrasse,5,Museum / Bibliothek,IMAX
79488,Lidostrasse,5,Museum / Bibliothek,Halle Luft- und Raumfahrt /
87201,Lidostrasse,5,Museum / Bibliothek,Halle Schienenverkehr 2 + 3
87204,Lidostrasse,7,Museum / Bibliothek,Hans Erni-Museum
87205,Lidostrasse,7,Museum / Bibliothek,Büroanbau Erni Museum


In [11]:
spital = filtered[filtered['GKLAS_TXT'] == 'Krankenhaus'].copy()

print('Krankenhaus:', len(spital))
display(spital[['STRNAMK1_HPT', 'DEINR', 'GKLAS_TXT', 'GBEZ']])

Krankenhaus: 13


,STRNAMK1_HPT,DEINR,GKLAS_TXT,GBEZ
71718,Lützelmattstrasse,1,Krankenhaus,UMNUTZUNG IN BÜRO
71719,Lützelmattstrasse,3,Krankenhaus,None
71995,Kantonsspital,31,Krankenhaus,Bettenhochhaus
79306,Kantonsspital,31,Krankenhaus,Verwaltungsgebäude
79685,Kantonsspital,30,Krankenhaus,Magnetresonanz-Tomographie
79782,St.-Anna-Strasse,36,Krankenhaus,Labor- und Bürogebäude Trakt E
79783,St.-Anna-Strasse,36,Krankenhaus,Trakt E und F
79784,St.-Anna-Strasse,34,Krankenhaus,Trakt D
81244,Kantonsspital,31,Krankenhaus,Spitalbau Breitfuss
83277,Kantonsspital,28,Krankenhaus,Strahlentherapie/Onkologie


In [12]:
gemeinschaft = filtered[filtered['GKLAS_TXT'] == 'Wohngeb.f.Gemeinschaften'].copy()

print('Wohngeb.f.Gemeinschaften:', len(gemeinschaft))
display(gemeinschaft[['STRNAMK1_HPT', 'DEINR', 'GKLAS_TXT', 'GBEZ']])

Wohngeb.f.Gemeinschaften: 22


,STRNAMK1_HPT,DEINR,GKLAS_TXT,GBEZ
23643,Staffelnhofstrasse,60,Wohngeb.f.Gemeinschaften,Alterszentrum
71725,Rigistrasse,48,Wohngeb.f.Gemeinschaften,Abzubrechen
71730,Tivolistrasse,21,Wohngeb.f.Gemeinschaften,Gemeinschaftszentrum St. Anna
24481,Tivolistrasse,5,Wohngeb.f.Gemeinschaften,Schwestern-Pflegerinnenheim
24505,Haldenstrasse,41,Wohngeb.f.Gemeinschaften,ABZUBRECHEN Personalhaus
25004,Kapuzinerweg,12,Wohngeb.f.Gemeinschaften,Pflegheim
25005,Kapuzinerweg,39,Wohngeb.f.Gemeinschaften,Kinderheim
25033,Wesemlinring,7,Wohngeb.f.Gemeinschaften,Kinderheim Titlisblick
25377,Rosenbergstrasse,2,Wohngeb.f.Gemeinschaften,Alterszentrum Rosenberg
25378,Rosenbergstrasse,4,Wohngeb.f.Gemeinschaften,Alterszentrum Rosenberg
